In [28]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import date
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

print("Libraries imported successfully!")

Libraries imported successfully!


In [31]:
DATA_DIR = Path("/internmo internship projects/project 2")

OUTPUT_DIR = Path("phase1_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

if not DATA_DIR.exists():
    raise FileNotFoundError("data folder nahi mila!")

print("Dataset folder:", DATA_DIR.resolve())

Dataset folder: D:\internmo internship projects\project 2


In [32]:
file_names = [
    "customers.csv",
    "subscriptions.csv",
    "plans.csv",
    "plan_history.csv",
    "contracts.csv",
    "devices.csv",
    "billing.csv",
    "payments.csv",
    "recharges.csv",
    "usage_voice.csv",
    "usage_sms.csv",
    "usage_data.csv",
    "network_quality.csv",
    "support_tickets.csv",
    "complaints.csv",
    "customer_feedback.csv",
    "retention_campaigns.csv",
    "marketing_campaigns.csv",
    "employees.csv",
    "stores.csv",
    "regions.csv",
    "cities.csv",
    "states.csv",
    "data_quality_issue_log.csv"
]

data = {}
missing_files = []

for file in file_names:
    path = DATA_DIR / file

    if path.exists():
        table_name = file.replace(".csv", "")
        data[table_name] = pd.read_csv(path, low_memory=False)
        print(f"{file} loaded: {data[table_name].shape}")
    else:
        missing_files.append(file)
        print(f"Missing file: {file}")

print("\nTotal loaded:", len(data))
print("Total missing:", len(missing_files))

customers.csv loaded: (19076, 25)
subscriptions.csv loaded: (20523, 8)
plans.csv loaded: (25, 10)
plan_history.csv loaded: (19349, 6)
contracts.csv loaded: (10607, 9)
devices.csv loaded: (19000, 8)
billing.csv loaded: (115313, 9)
payments.csv loaded: (92362, 7)
recharges.csv loaded: (146501, 6)
usage_voice.csv loaded: (57000, 6)
usage_sms.csv loaded: (57000, 5)
usage_data.csv loaded: (57000, 5)
network_quality.csv loaded: (57000, 8)
support_tickets.csv loaded: (31490, 11)
complaints.csv loaded: (16584, 7)
customer_feedback.csv loaded: (28499, 7)
retention_campaigns.csv loaded: (6475, 8)
marketing_campaigns.csv loaded: (40, 9)
employees.csv loaded: (800, 9)
stores.csv loaded: (2180, 7)
regions.csv loaded: (4, 3)
cities.csv loaded: (68, 8)
states.csv loaded: (36, 4)
data_quality_issue_log.csv loaded: (20, 3)

Total loaded: 24
Total missing: 0


In [33]:
profile = []

for name, df in data.items():
    profile.append({
        "Table": name,
        "Rows": len(df),
        "Columns": len(df.columns),
        "Duplicate_Rows": df.duplicated().sum(),
        "Memory_MB": round(
            df.memory_usage(deep=True).sum() / (1024**2), 2
        )
    })

profile_df = pd.DataFrame(profile)

display(profile_df)

,Table,Rows,Columns,Duplicate_Rows,Memory_MB
0,customers,19076,25,76,21.70
1,subscriptions,20523,8,0,7.50
2,plans,25,10,0,0.01
3,plan_history,19349,6,0,6.32
4,contracts,10607,9,0,4.71
5,devices,19000,8,0,6.52
6,billing,115313,9,344,40.58
7,payments,92362,7,276,31.29
8,recharges,146501,6,0,40.83
9,usage_voice,57000,6,0,10.65


In [34]:
missing_report = []

for name, df in data.items():
    for column in df.columns:
        missing_count = df[column].isnull().sum()

        missing_report.append({
            "Table": name,
            "Column": column,
            "Missing_Count": missing_count,
            "Missing_Percentage": round(
                (missing_count / len(df)) * 100, 2
            ) if len(df) > 0 else 0
        })

missing_df = pd.DataFrame(missing_report)

display(
    missing_df[missing_df["Missing_Count"] > 0]
    .sort_values("Missing_Percentage", ascending=False)
)

,Table,Column,Missing_Count,Missing_Percentage
29,subscriptions,end_date,17017,82.92
24,customers,churn_date,15560,81.57
129,complaints,resolution_date,7348,44.31
122,support_tickets,resolution_date,13366,42.45
121,support_tickets,resolution_hours,13366,42.45
16,customers,annual_income_inr,954,5.00
6,customers,email,745,3.91
8,customers,address,380,1.99
27,subscriptions,plan_id,41,0.20


In [35]:
print("Exact duplicate rows in each table:\n")

for name, df in data.items():
    print(name, ":", df.duplicated().sum())

Exact duplicate rows in each table:

customers : 76
subscriptions : 0
plans : 0
plan_history : 0
contracts : 0
devices : 0
billing : 344
payments : 276
recharges : 0
usage_voice : 0
usage_sms : 0
usage_data : 0
network_quality : 0
support_tickets : 0
complaints : 0
customer_feedback : 0
retention_campaigns : 0
marketing_campaigns : 0
employees : 0
stores : 0
regions : 0
cities : 0
states : 0
data_quality_issue_log : 0


In [36]:
primary_keys = {
    "customers": "customer_id",
    "subscriptions": "subscription_id",
    "billing": "invoice_id",
    "payments": "payment_id",
    "complaints": "complaint_id",
    "support_tickets": "ticket_id"
}

duplicate_keys = []

for table, column in primary_keys.items():

    if table not in data:
        continue

    df = data[table]

    if column not in df.columns:
        print(f"{column} column not found in {table}")
        continue

    duplicate_count = df[column].duplicated().sum()
    null_count = df[column].isnull().sum()

    duplicate_keys.append({
        "Table": table,
        "Key": column,
        "Duplicate_Count": duplicate_count,
        "Null_Count": null_count
    })

duplicate_key_df = pd.DataFrame(duplicate_keys)
display(duplicate_key_df)

,Table,Key,Duplicate_Count,Null_Count
0,customers,customer_id,76,0
1,subscriptions,subscription_id,0,0
2,billing,invoice_id,344,0
3,payments,payment_id,276,0
4,complaints,complaint_id,0,0
5,support_tickets,ticket_id,0,0


In [37]:
working = {
    name: df.copy()
    for name, df in data.items()
}

date_columns = {
    "customers": ["acquisition_date", "churn_date"],
    "billing": ["billing_date"],
    "payments": ["payment_date"],
    "subscriptions": ["start_date", "end_date"],
    "plan_history": ["change_date"],
    "contracts": ["start_date", "end_date"],
    "recharges": ["recharge_date"]
}

date_issues = []

for table, columns in date_columns.items():

    if table not in working:
        continue

    df = working[table]

    for column in columns:

        if column not in df.columns:
            continue

        original = df[column].copy()

        parsed = pd.to_datetime(
            original,
            format="%d-%m-%Y",
            errors="coerce"
        )

        # Fallback for valid dates written in another format
        failed = original.notna() & parsed.isna()

        if failed.any():
            parsed.loc[failed] = pd.to_datetime(
                original.loc[failed],
                errors="coerce",
                dayfirst=True
            )

        invalid_count = (
            original.notna() & parsed.isna()
        ).sum()

        date_issues.append({
            "Table": table,
            "Column": column,
            "Invalid_Dates": int(invalid_count)
        })

        df[column] = parsed

date_issues_df = pd.DataFrame(date_issues)
display(date_issues_df)

C:\Users\harsh\AppData\Local\Temp\ipykernel_3512\678241580.py:42: UserWarning: Parsing dates in %Y/%m/%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  parsed.loc[failed] = pd.to_datetime(


,Table,Column,Invalid_Dates
0,customers,acquisition_date,0
1,customers,churn_date,0
2,billing,billing_date,0
3,payments,payment_date,0
4,subscriptions,start_date,0
5,subscriptions,end_date,0
6,plan_history,change_date,0
7,contracts,start_date,0
8,contracts,end_date,0
9,recharges,recharge_date,0


In [38]:
numeric_columns = {
    "customers": ["tenure_months", "annual_income_inr"],
    "subscriptions": ["monthly_charge_inr"],
    "billing": [
        "base_amount_inr",
        "gst_amount_inr",
        "total_amount_inr"
    ],
    "usage_data": ["data_gb_used", "data_sessions"],
    "support_tickets": ["resolution_hours"]
}

numeric_issues = []

for table, columns in numeric_columns.items():

    if table not in working:
        continue

    df = working[table]

    for column in columns:

        if column not in df.columns:
            continue

        original = df[column].copy()
        converted = pd.to_numeric(
            original,
            errors="coerce"
        )

        invalid_count = (
            original.notna() & converted.isna()
        ).sum()

        numeric_issues.append({
            "Table": table,
            "Column": column,
            "Invalid_Numeric_Values": int(invalid_count)
        })

        df[column] = converted

numeric_issues_df = pd.DataFrame(numeric_issues)
display(numeric_issues_df)

,Table,Column,Invalid_Numeric_Values
0,customers,tenure_months,0
1,customers,annual_income_inr,0
2,subscriptions,monthly_charge_inr,0
3,billing,base_amount_inr,0
4,billing,gst_amount_inr,0
5,billing,total_amount_inr,0
6,usage_data,data_gb_used,0
7,usage_data,data_sessions,0
8,support_tickets,resolution_hours,0


In [39]:
if "billing" in working:

    billing = working["billing"]

    if "billing_date" in billing.columns:

        today = pd.Timestamp(date.today())

        future_bills = billing[
            billing["billing_date"] > today
        ]

        print("Future billing dates:", len(future_bills))
        display(future_bills.head(10))

    if "total_amount_inr" in billing.columns:

        negative_bills = billing[
            billing["total_amount_inr"] < 0
        ]

        print("Negative invoice amounts:", len(negative_bills))
        display(negative_bills.head(10))

Future billing dates: 112


,invoice_id,customer_id,billing_date,billing_period_month,base_amount_inr,gst_amount_inr,total_amount_inr,due_date,payment_status
1584,INV00001585,C0000279,2027-04-11,05-2025,599.40,107.89,707.29,20-05-2025,Unpaid
2829,INV00002830,C0000512,2027-01-30,01-2026,897.03,161.47,1058.50,14-02-2026,Paid
3010,INV00003011,C0000549,2027-02-01,03-2026,898.31,161.70,1060.01,15-04-2026,Paid
3059,INV00003060,C0000556,2026-12-11,03-2026,500.58,90.10,590.68,15-04-2026,Paid
8746,INV00008747,C0001532,2027-04-19,02-2026,497.85,89.61,587.46,17-02-2026,Paid
9101,INV00009102,C0001598,2027-04-25,08-2025,1085.04,195.31,1280.35,18-08-2025,Unpaid
9175,INV00009176,C0001611,2026-12-01,06-2024,474.02,85.32,559.34,26-06-2024,Paid
9953,INV00009954,C0001750,2026-12-01,07-2023,1306.21,235.12,1541.33,16-07-2023,Unpaid
10134,INV00010135,C0001777,2026-11-15,05-2025,1022.62,184.07,1206.69,20-05-2025,Paid
10554,INV00010555,C0001844,2026-11-12,07-2025,697.67,125.58,823.25,19-07-2025,Unpaid


Negative invoice amounts: 229


,invoice_id,customer_id,billing_date,billing_period_month,base_amount_inr,gst_amount_inr,total_amount_inr,due_date,payment_status
352,INV00000353,C0000074,2025-12-01,12-2025,611.62,110.09,-721.71,16-12-2025,Paid
559,INV00000560,C0000102,2026-03-01,03-2026,639.46,115.10,-754.56,16-03-2026,Paid
716,INV00000717,C0000124,2025-09-02,09-2025,1324.07,238.33,-1562.40,17-09-2025,Overdue
867,INV00000868,C0000156,2025-10-02,10-2025,1408.69,253.56,-1662.25,17-10-2025,Paid
1030,INV00001031,C0000192,2026-03-31,03-2026,531.10,95.60,-626.70,15-04-2026,Unpaid
1900,INV00001901,C0000331,2026-03-01,03-2026,1519.83,273.57,-1793.40,16-03-2026,Paid
3810,INV00003811,C0000696,2025-12-31,12-2025,3909.14,703.65,-4612.79,15-01-2026,Unpaid
4479,INV00004480,C0000829,2025-08-03,08-2025,1515.23,272.74,-1787.97,18-08-2025,Paid
4634,INV00004635,C0000848,2025-09-02,09-2025,473.07,85.15,-558.22,17-09-2025,Paid
5094,INV00005095,C0000923,2025-09-02,09-2025,784.81,141.27,-926.08,17-09-2025,Overdue


In [40]:
if "usage_data" in working:

    usage = working["usage_data"]

    if "data_gb_used" in usage.columns:

        values = usage["data_gb_used"].dropna()

        Q1 = values.quantile(0.25)
        Q3 = values.quantile(0.75)

        IQR = Q3 - Q1

        lower_limit = Q1 - 1.5 * IQR
        upper_limit = Q3 + 1.5 * IQR

        usage["outlier_flag"] = (
            (usage["data_gb_used"] < lower_limit) |
            (usage["data_gb_used"] > upper_limit)
        )

        outliers = usage[usage["outlier_flag"]]

        print("Q1:", Q1)
        print("Q3:", Q3)
        print("Lower limit:", lower_limit)
        print("Upper limit:", upper_limit)
        print("Total outliers:", len(outliers))

        display(
            outliers.sort_values(
                "data_gb_used",
                ascending=False
            ).head(20)
        )

Q1: 11.39
Q3: 32.63
Lower limit: -20.470000000000002
Upper limit: 64.49000000000001
Total outliers: 4480


,usage_id,customer_id,usage_month,data_gb_used,data_sessions,outlier_flag
15077,UD00015078,C0005026,01-2026,22172.700708,31,True
23089,UD00023090,C0007697,02-2026,21960.635745,489,True
10054,UD00010055,C0003352,02-2026,18348.446462,369,True
34859,UD00034860,C0011620,01-2026,17377.228312,765,True
26592,UD00026593,C0008865,03-2026,15322.701632,268,True
29837,UD00029838,C0009946,01-2026,13886.727541,1105,True
53991,UD00053992,C0017998,03-2026,13344.522727,862,True
56096,UD00056097,C0018699,01-2026,13154.906387,317,True
45208,UD00045209,C0015070,02-2026,12747.455634,929,True
19929,UD00019930,C0006644,03-2026,12581.380386,496,True


In [41]:
if "customers" in working:

    customers = working["customers"]

    # Age validation
    age_col = next(
        (c for c in ["age", "customer_age", "age_years"]
         if c in customers.columns),
        None
    )

    if age_col:
        age = pd.to_numeric(
            customers[age_col],
            errors="coerce"
        )

        invalid_age = customers[
            age.notna() & ((age < 0) | (age > 120))
        ]

        print("Invalid ages:", len(invalid_age))
        display(invalid_age.head(10))
    else:
        print("Age column available nahi hai.")

    # PIN code validation
    pin_col = next(
        (c for c in ["pin_code", "pincode", "postal_code"]
         if c in customers.columns),
        None
    )

    if pin_col:
        pin = customers[pin_col].astype("string").str.strip()

        invalid_pin = customers[
            pin.notna() & ~pin.str.fullmatch(r"\d{6}", na=False)
        ]

        print("Invalid PIN codes:", len(invalid_pin))
        display(invalid_pin.head(10))
    else:
        print("PIN code column available nahi hai.")

    # Phone validation
    phone_col = next(
        (c for c in ["phone_number", "mobile_number", "phone", "mobile"]
         if c in customers.columns),
        None
    )

    if phone_col:
        phone = (
            customers[phone_col]
            .astype("string")
            .str.replace(r"\D", "", regex=True)
        )

        valid_phone = phone.str.fullmatch(
            r"(?:91)?[6-9]\d{9}",
            na=False
        )

        invalid_phone = customers[
            customers[phone_col].notna() & ~valid_phone
        ]

        print("Invalid phone numbers:", len(invalid_phone))
        display(invalid_phone.head(10))
    else:
        print("Phone column available nahi hai.")

Invalid ages: 23


,customer_id,first_name,last_name,gender,date_of_birth,age,email,phone_number,address,city_id,city_name,state_id,pin_code,city_tier,region_id,occupation,annual_income_inr,customer_segment,product_line,plan_id,acquisition_channel,acquisition_date,tenure_months,customer_status,churn_date
751,C0000752,Anika,Pandey,Female,31-03-1989,220,anika.pandey265@rediffmail.com,916763149388,"Villa 136, Vivekananda Phase, Chennai",CT0004,Chennai,ST016,600743,Metro,RG03,Farmer,129668.0,Mass,Prepaid Mobile,PL001,Retail Store,2023-05-14,34,Active,NaT
834,C0000835,Sridevi,Sen,Female,31-03-1988,220,sridevi.sen783@gmail.com,916550906093,"Door No. 72, New Phase, Tirupati",CT0042,Tirupati,ST018,517281,Tier-3,RG03,Retired,238954.0,Value,Prepaid Mobile,PL004,Referral,2018-07-26,92,Active,NaT
2917,C0002918,Geeta,Qureshi,Female,31-03-1984,130,geeta.qureshi848@rediffmail.com,917236699438,"House No. 328, Vivekananda Street, Kollam",CT0046,Kollam,ST017,691970,Tier-3,RG03,Bank Officer,NaN,Value,Prepaid Mobile,PL005,Online,2014-01-01,146,Active,NaT
3026,C0003027,Aarav,Shaikh,Male,01-04-1971,220,aarav.shaikh942@hotmail.com,916255560170,"House No. 66, Janpath Phase, Amritsar",CT0032,Amritsar,ST003,3782,Tier-2,RG01,Bank Officer,1049990.0,Enterprise,IoT Solutions,PL024,Online,2023-09-14,30,Active,NaT
3822,C0003823,Zara,Trivedi,Female,01-04-1983,220,zara.trivedi587@yahoo.in,15145,"Villa 12, Park View Colony, Gwalior",CT0048,Gwalior,ST012,474731,Tier-3,RG02,HR Executive,376701.0,Value,Fiber Broadband,PL015,Retail Store,2014-01-01,146,Active,NaT
3824,C0003825,Saanvi,Aggarwal,Female,01-04-1979,220,saanvi.aggarwal636@outlook.com,919008185404,"Door No. 252, Vasant Marg, Mumbai",CT0002,Mumbai,ST010,400503,Metro,RG02,Marketing Manager,2047567.0,Premium,Postpaid Mobile,PL009,Referral,2022-03-26,48,Active,NaT
4639,C0004640,Tarun,Khanna,Male,31-03-1982,130,tarun.khanna43@hotmail.com,916853291796,"Apartment 241, Park View Cross, Belagavi",CT0041,Belagavi,ST015,590800,Tier-3,RG03,Chartered Accountant,2818203.0,Mass,Prepaid Mobile,PL001,Partner/Reseller,2023-08-08,31,Active,NaT
5159,C0005160,Kavitha,Nayak,Female,31-03-1969,130,kavitha.nayak409@rediffmail.com,919528137142,"Flat 371, Vasant Road, Shillong",CT0065,Shillong,ST034,793443,Tier-3,RG04,Doctor,1948773.0,Premium,5G,PL012,Tele-sales,2020-04-10,71,Active,NaT
5290,C0005291,Kiara,More,Female,31-03-2006,130,kiara.more1@gmail.com,916338889455,"Villa 347, Garden Sector, Nashik",CT0030,Nashik,ST010,422551,Tier-2,RG02,Operations Manager,NaN,Premium,Prepaid Mobile,PL006,Retail Store,2014-01-01,146,Active,NaT
5833,C0005834,Venkat,Dutta,Male,31-03-1982,-5,venkat.dutta564@yahoo.in,916745884801,"Shop No. 316, New Colony, Jodhpur",CT0035,Jodhpur,ST005,342936,Tier-2,RG01,HR Executive,423955.0,Premium,Fiber Broadband,PL017,Retail Store,2023-09-14,30,Active,NaT


Invalid PIN codes: 191


,customer_id,first_name,last_name,gender,date_of_birth,age,email,phone_number,address,city_id,city_name,state_id,pin_code,city_tier,region_id,occupation,annual_income_inr,customer_segment,product_line,plan_id,acquisition_channel,acquisition_date,tenure_months,customer_status,churn_date
45,C0000046,Suresh,Borah,Male,31-03-1988,38,suresh.borah334@hotmail.com,919980790921,"Shop No. 350, Civil Lines Road, Chandigarh",CT0023,Chandigarh,ST009,9172,Tier-2,RG01,Government Employee,415706.0,Premium,Prepaid Mobile,PL006,Retail Store,2014-01-01,146,Active,NaT
182,C0000183,Arjun,Nayak,Male,31-03-1992,34,arjun.nayak409@hotmail.com,917679804098,"Door No. 316, Rajaji Lane, Lucknow",CT0012,Lucknow,ST004,7123,Tier-2,RG01,Farmer,266783.0,Value,Prepaid Mobile,PL005,Retail Store,2024-06-20,21,Active,NaT
185,C0000186,Manish,Borah,Male,31-03-2005,21,manish.borah745@hotmail.com,918188485280,"Apartment 174, MG Marg, Durgapur",CT0053,Durgapur,ST022,7128,Tier-3,RG04,Sales Executive,379870.0,Value,OTT Bundles,PL019,App Self-Service,2021-01-05,62,Active,NaT
360,C0000361,Abhishek,Menon,Male,31-03-1978,48,abhishek.menon429@gmail.com,919386564980,"Shop No. 149, Subhash Marg, Dehradun",CT0033,Dehradun,ST006,4841,Tier-2,RG01,Software Engineer,1938608.0,Premium,5G,PL012,Tele-sales,2025-03-29,11,Churned,2026-03-01
517,C0000518,Imran,More,Male,31-03-1997,29,imran.more458@gmail.com,916567981144,"Plot 197, Nehru Sector, Ludhiana",CT0031,Ludhiana,ST003,8109,Tier-2,RG01,Sales Executive,606996.0,Premium,Postpaid Mobile,PL008,Online,2015-07-01,128,Active,NaT
534,C0000535,Nandini,Qureshi,Female,31-03-1993,33,nandini.qureshi409@hotmail.com,918487556208,"Plot 308, Vivekananda Road, Bhagalpur",CT0056,Bhagalpur,ST024,2180,Tier-3,RG04,Retail Shopkeeper,373569.0,Mass,Prepaid Mobile,PL002,Partner/Reseller,2022-01-26,50,Active,NaT
586,C0000587,Harish,Solanki,Male,31-03-1993,33,harish.solanki377@hotmail.com,917931226476,"Plot 111, Subhash Lane, Chennai",CT0004,Chennai,ST016,1470,Metro,RG03,Farmer,210025.0,Premium,5G,PL012,Online,2024-02-22,25,Active,NaT
610,C0000611,Geeta,Iyer,Female,31-03-1997,29,NaN,919310088046,"Plot 320, Industrial Cross, Lucknow",CT0012,Lucknow,ST004,5514,Tier-2,RG01,HR Executive,792071.0,Premium,Fiber Broadband,PL017,Referral,2014-01-01,146,Active,NaT
673,C0000674,Rohan,Chandran,Male,31-03-1988,38,rohan.chandran351@yahoo.in,918163907075,"Shop No. 354, Patel Colony, Noida",CT0010,Noida,ST004,9869,Metro,RG01,Doctor,1632988.0,Value,Prepaid Mobile,PL003,Online,2020-06-08,69,Active,NaT
679,C0000680,Sneha,Sen,Female,31-03-2005,21,sneha.sen461@outlook.com,916232532569,"Plot 15, Lakshmi Marg, Noida",CT0010,Noida,ST004,6401,Metro,RG01,Auto/Cab Driver,376357.0,Premium,Postpaid Mobile,PL008,Retail Store,2020-06-01,69,Active,NaT


Invalid phone numbers: 228


,customer_id,first_name,last_name,gender,date_of_birth,age,email,phone_number,address,city_id,city_name,state_id,pin_code,city_tier,region_id,occupation,annual_income_inr,customer_segment,product_line,plan_id,acquisition_channel,acquisition_date,tenure_months,customer_status,churn_date
10,C0000011,Pankaj,Joshi,Male,01-04-1979,47,pankaj.joshi32@outlook.com,57059,"Flat 342, New Phase, Mysuru",CT0027,Mysuru,ST015,570685,Tier-2,RG03,Auto/Cab Driver,228930.0,Value,Prepaid Mobile,PL004,Tele-sales,2015-06-08,129,Active,NaT
65,C0000066,Ramesh,More,Male,01-04-1983,43,ramesh.more888@gmail.com,13624,"Villa 358, Rajaji Colony, Patna",CT0021,Patna,ST024,800696,Tier-2,RG04,Sales Executive,789784.0,Premium,5G,PL014,Tele-sales,2020-08-11,1,Churned,2020-09-22
138,C0000139,Sneha,Khanna,Female,31-03-1989,37,sneha.khanna906@yahoo.in,91566,"Villa 102, Krishna Layout, Bengaluru",CT0003,BENGALURU,ST015,560723,Metro,RG03,Small Business Owner,1221600.0,Mass,Prepaid Mobile,PL001,Partner/Reseller,2017-12-15,90,Churned,2025-07-02
404,C0000405,Faisal,Vyas,Male,31-03-1973,53,faisal.vyas931@gmail.com,75835,"Flat 387, Old Lane, Mumbai",CT0002,Mumbai,ST010,400952,Metro,RG02,Bank Officer,878931.0,Premium,5G,PL014,Online,2024-11-29,8,Churned,2025-08-29
405,C0000406,Yash,Naidu,Male,01-04-1971,55,yash.naidu237@yahoo.in,55071,"Door No. 322, Model Town Street, Thiruvanantha...",CT0034,Thiruvananthapuram,ST017,695982,Tier-2,RG03,Chartered Accountant,1729704.0,Premium,Postpaid Mobile,PL008,Online,2023-04-29,25,Churned,2025-05-29
524,C0000525,Aakash,Raju,Male,01-04-1991,35,aakash.raju917@yahoo.in,70233,"Villa 108, Indira Street, Pune",CT0007,Pune,ST010,411657,Metro,RG02,Auto/Cab Driver,370778.0,Enterprise,Enterprise,PL022,App Self-Service,2018-03-14,96,Active,NaT
544,C0000545,Saanvi,Bhardwaj,Female,31-03-1985,41,saanvi.bhardwaj850@yahoo.in,84892,"Flat 378, Gandhi Phase, Patna",CT0021,Patna,ST024,800957,Tier-2,RG04,School Teacher,465177.0,Premium,Prepaid Mobile,PL006,Referral,2021-05-30,58,Active,NaT
579,C0000580,Vihaan,Subramanian,Male,31-03-1982,44,vihaan.subramanian741@yahoo.in,24255,"Flat 220, Shivaji Marg, Hyderabad",CT0006,Hyderabad,ST019,500936,Metro,RG03,Software Engineer,1760430.0,Premium,5G,PL013,Partner/Reseller,2025-09-25,3,Churned,2025-12-31
585,C0000586,Kavitha,Bose,Female,31-03-2008,18,kavitha.bose641@outlook.com,97577,"Door No. 160, Janpath Road, Kanpur",CT0036,Kanpur,ST004,208235,Tier-2,RG01,Auto/Cab Driver,403495.0,Enterprise,IoT Solutions,PL023,Online,2016-03-22,120,Active,NaT
663,C0000664,Reyansh,Kammela,Male,01-04-1959,67,reyansh.kammela624@rediffmail.com,47167,"Door No. 339, Lakshmi Colony, Ahmedabad",CT0008,Ahmedabad,ST011,380613,Metro,RG02,Government Employee,974058.0,Premium,Postpaid Mobile,PL010,Online,2017-05-20,106,Active,NaT


In [42]:
integrity_results = []

if "customers" in working and "customer_id" in working["customers"].columns:

    valid_customer_ids = set(
        working["customers"]["customer_id"]
        .dropna()
        .astype(str)
        .str.strip()
    )

    tables_to_check = [
        "subscriptions",
        "billing",
        "usage_voice",
        "usage_sms",
        "usage_data",
        "network_quality"
    ]

    for table in tables_to_check:

        if table not in working:
            continue

        df = working[table]

        if "customer_id" not in df.columns:
            continue

        ids = df["customer_id"]

        invalid_mask = (
            ids.notna() &
            ~ids.astype(str).str.strip().isin(valid_customer_ids)
        )

        integrity_results.append({
            "Table": table,
            "Rows_Checked": int(ids.notna().sum()),
            "Broken_References": int(invalid_mask.sum())
        })

        print(f"\n{table}: broken references = {invalid_mask.sum()}")
        display(df.loc[invalid_mask].head(5))

integrity_df = pd.DataFrame(integrity_results)
display(integrity_df)


subscriptions: broken references = 0


,subscription_id,customer_id,plan_id,start_date,end_date,billing_cycle,monthly_charge_inr,subscription_status



billing: broken references = 0


,invoice_id,customer_id,billing_date,billing_period_month,base_amount_inr,gst_amount_inr,total_amount_inr,due_date,payment_status



usage_voice: broken references = 0


...


usage_data: broken references = 0


,usage_id,customer_id,usage_month,data_gb_used,data_sessions,outlier_flag



network_quality: broken references = 0


,record_id,customer_id,city_id,month,avg_signal_strength_dbm,dropped_call_rate_pct,avg_download_speed_mbps,network_downtime_hours


,Table,Rows_Checked,Broken_References
0,subscriptions,20523,0
1,billing,115313,0
2,usage_voice,57000,0
3,usage_sms,57000,0
4,usage_data,57000,0
5,network_quality,57000,0


In [43]:
if "payments" in working and "billing" in working:

    payments = working["payments"]
    billing = working["billing"]

    if "invoice_id" in payments.columns and "invoice_id" in billing.columns:

        valid_invoice_ids = set(
            billing["invoice_id"].dropna().astype(str).str.strip()
        )

        payment_invoice_ids = payments["invoice_id"]

        broken_payments = payments[
            payment_invoice_ids.notna() &
            ~payment_invoice_ids.astype(str)
            .str.strip()
            .isin(valid_invoice_ids)
        ]

        print("Total payments:", len(payments))
        print("Broken invoice references:", len(broken_payments))

        display(broken_payments.head(10))

Total payments: 92362
Broken invoice references: 184


,payment_id,invoice_id,customer_id,payment_date,amount_inr,payment_method,payment_status
874,PAY00000875,INV94560686,C0000199,2026-01-11,1444.45,Credit Card,Success
1007,PAY00001008,INV96995702,C0000232,2025-12-27,723.82,Credit Card,Success
1474,PAY00001475,INV99552429,C0000319,2025-05-24,1439.82,Credit Card,Success
2112,PAY00002113,INV98863464,C0000473,2025-07-08,746.33,UPI,Success
2117,PAY00002118,INV92113763,C0000477,2025-11-07,1536.96,Cash,Success
2380,PAY00002381,INV92607579,C0000533,2025-09-06,817.30,Wallet,Success
2759,PAY00002760,INV98698649,C0000628,2025-08-11,850.38,Auto Debit,Success
3251,PAY00003252,INV98048665,C0000733,2026-02-08,343.09,Net Banking,Success
3589,PAY00003590,INV97878268,C0000827,2026-02-05,694.26,IMPS,Success
3937,PAY00003938,INV97615652,C0000892,2024-05-10,1483.26,Net Banking,Success


In [44]:
if "subscriptions" in working:

    subs = working["subscriptions"]

    if "plan_id" in subs.columns:
        print("Null plan_id:", subs["plan_id"].isna().sum())

    if "subscription_status" in subs.columns:

        valid_statuses = ["active", "terminated"]

        status = (
            subs["subscription_status"]
            .astype("string")
            .str.strip()
            .str.lower()
        )

        invalid_status = subs[
            status.notna() & ~status.isin(valid_statuses)
        ]

        print("Invalid subscription statuses:", len(invalid_status))
        display(invalid_status.head(10))

if "complaints" in working and "status" in working["complaints"].columns:

    complaints = working["complaints"]

    status = (
        complaints["status"]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    unresolved_count = status.eq("unresolved").sum()
    valid_status_count = status.notna().sum()

    unresolved_percentage = (
        unresolved_count / valid_status_count * 100
        if valid_status_count > 0 else 0
    )

    print("Total complaints:", len(complaints))
    print("Unresolved complaints:", unresolved_count)
    print(f"Unresolved percentage: {unresolved_percentage:.2f}%")
    

Null plan_id: 41
Invalid subscription statuses: 102


,subscription_id,customer_id,plan_id,start_date,end_date,billing_cycle,monthly_charge_inr,subscription_status
66,S0000067,C0000067,PL006,2014-01-01,NaT,Annual,300.0,UNKNOWN_STATE
74,S0000075,C0000075,PL005,2024-12-13,NaT,Annual,250.0,UNKNOWN_STATE
128,S0000129,C0000129,PL005,2014-01-01,2018-10-07,Annual,250.0,UNKNOWN_STATE
185,S0000186,C0000186,PL019,2021-01-05,NaT,Monthly,349.0,UNKNOWN_STATE
188,S0000189,C0000189,PL006,2015-08-02,NaT,Annual,300.0,UNKNOWN_STATE
212,S0000213,C0000213,PL009,2021-10-19,NaT,Monthly,999.0,UNKNOWN_STATE
905,S0000906,C0000906,PL008,2023-08-15,NaT,Monthly,699.0,UNKNOWN_STATE
979,S0000980,C0000980,PL006,2017-08-15,NaT,Annual,300.0,UNKNOWN_STATE
1332,S0001333,C0001333,PL009,2025-06-01,NaT,Monthly,999.0,UNKNOWN_STATE
1574,S0001575,C0001575,PL015,2015-07-23,NaT,Monthly,699.0,UNKNOWN_STATE


Total complaints: 16584
Unresolved complaints: 7348
Unresolved percentage: 44.31%


In [45]:
if "data_quality_issue_log" in working:

    dq_log = working["data_quality_issue_log"]

    print("Issue log shape:", dq_log.shape)
    print("Columns:", dq_log.columns.tolist())

    display(dq_log)

else:
    print("data_quality_issue_log.csv load nahi hui.")

Issue log shape: (20, 3)
Columns: ['table', 'issue_type', 'rows_affected']


,table,issue_type,rows_affected
0,customers,missing_email,760
1,customers,missing_address,380
2,customers,missing_income,950
3,customers,invalid_phone,228
4,customers,invalid_email,285
5,customers,incorrect_pin,190
6,customers,misspelled_city,380
7,customers,inconsistent_capitalisation,570
8,customers,impossible_age,38
9,customers,mixed_date_format,380


In [46]:
reports = {
    "table_profile.csv": profile_df,
    "missing_values_report.csv": missing_df,
    "duplicate_keys_report.csv": duplicate_key_df,
    "date_issues_report.csv": date_issues_df,
    "numeric_issues_report.csv": numeric_issues_df,
    "referential_integrity_report.csv": integrity_df
}

for filename, report in reports.items():

    report.to_csv(
        OUTPUT_DIR / filename,
        index=False
    )

    print("Saved:", filename)

print("\nAll quality reports exported successfully!")

Saved: table_profile.csv
Saved: missing_values_report.csv
Saved: duplicate_keys_report.csv
Saved: date_issues_report.csv
Saved: numeric_issues_report.csv
Saved: referential_integrity_report.csv

All quality reports exported successfully!


In [47]:
total_rows = sum(len(df) for df in data.values())

tables_with_missing = (
    missing_df.loc[
        missing_df["Missing_Count"] > 0, "Table"
    ].nunique()
)

total_duplicates = sum(
    df.duplicated().sum()
    for df in data.values()
)

report = f"""
NEXATEL CUSTOMER CHURN ANALYTICS
PHASE 1: DATA QUALITY REPORT

Prepared by: Harsh Suman
Report Date: {date.today().strftime("%d-%m-%Y")}

1. DATA LOADING
Expected CSV files: {len(file_names)}
Successfully loaded: {len(data)}
Missing files: {len(missing_files)}
Total rows across loaded tables: {total_rows:,}

2. DATA QUALITY FINDINGS
Tables containing missing values: {tables_with_missing}
Total exact duplicate rows across tables: {total_duplicates}

3. REFERENTIAL INTEGRITY
Relationships checked: {len(integrity_df)}
Review referential_integrity_report.csv for broken references.

4. DATA VALIDATION
Review date_issues_report.csv for invalid dates.
Review numeric_issues_report.csv for invalid numeric values.
Review billing checks for future dates and negative amounts.
Review usage outlier flags for unusual data consumption.
Review customer validation and subscription checks.

5. RECOMMENDATIONS
- Investigate missing values before deciding how to handle them.
- Verify duplicate primary keys before removing any records.
- Review unusual usage values with business context.
- Correct invalid dates and numeric values only after verification.
- Compare detected problems with data_quality_issue_log.csv.
- Preserve the original raw CSV files.

Note: All findings should be verified against the actual dataset
before this report is submitted.
"""

print(report)

with open(
    OUTPUT_DIR / "Phase1_Data_Quality_Report.txt",
    "w",
    encoding="utf-8"
) as file:
    file.write(report)

print("Quality report saved successfully!")


NEXATEL CUSTOMER CHURN ANALYTICS
PHASE 1: DATA QUALITY REPORT

Prepared by: Harsh Suman
Report Date: 11-10-2026

1. DATA LOADING
Expected CSV files: 24
Successfully loaded: 24
Missing files: 0
Total rows across loaded tables: 756,952

2. DATA QUALITY FINDINGS
Tables containing missing values: 4
Total exact duplicate rows across tables: 696

3. REFERENTIAL INTEGRITY
Relationships checked: 6
Review referential_integrity_report.csv for broken references.

4. DATA VALIDATION
Review date_issues_report.csv for invalid dates.
Review numeric_issues_report.csv for invalid numeric values.
Review billing checks for future dates and negative amounts.
Review usage outlier flags for unusual data consumption.
Review customer validation and subscription checks.

5. RECOMMENDATIONS
- Investigate missing values before deciding how to handle them.
- Verify duplicate primary keys before removing any records.
- Review unusual usage values with business context.
- Correct invalid dates and numeric values o